# 빵셀러 종합 프로젝트 — 데이터 정제·집계

베이커리 체인 "빵셀러"의 가상 운영 데이터를 생성하고, 품질 점검 → 정제 파이프라인 → KPI 집계 → 저장 → 보고서 순서로 진행합니다.

| 순서 | 내용 |
|---|---|
| 0 | 환경 준비 및 데이터 생성 |
| 1 | `quality_report_full()` 품질 점검 |
| 2 | 정제 함수 정의와 `.pipe()` 파이프라인 |
| 3 | 결정 로그 표 |
| 4 | KPI 집계 (월·매장 / 상품) |
| 5 | Parquet·CSV 저장과 용량 비교 |
| 6 | 종합 프로젝트 보고서 |

## 0. 환경 준비 및 데이터 생성

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import pyarrow
print(pyarrow.__version__)

pd.set_option("display.max_columns", 30)
OUT = Path("output")
OUT.mkdir(exist_ok=True)

25.0.0


In [2]:
# 종합 프로젝트용 새 데이터 생성 — 베이커리 체인 "빵셀러" 운영 데이터
np.random.seed(13)
n_days = 90
n_stores = 6
n_rows = n_days * n_stores * 4   # 매장 x 일자 x 시간대(4구간)

stores = [f"S{i:02d}" for i in range(1, n_stores + 1)]
items = ["크루아상", "식빵", "케이크", "샌드위치", "쿠키"]

bakery = pd.DataFrame({
    "store_id": np.tile(stores, n_rows // n_stores)[:n_rows],
    "date_str": np.random.choice([
        "2025-04-01", "2025/04/01", "20250401",
        "2025-05-15", "2025/05/15", "20250515",
        "2025-06-20", "2025/06/20", "20250620",
    ], n_rows),
    "item": np.random.choice(items, n_rows),
    "quantity": np.random.choice([1, 1, 2, 2, 3, 5, 50], n_rows),  # 50은 의심값
    "unit_price": np.random.choice([3500, 4500, 5500, 8500, 12000], n_rows),
})
bakery["revenue"] = bakery["quantity"] * bakery["unit_price"]

# 오염 심기
bakery.loc[np.random.choice(n_rows, 60, replace=False), "revenue"] = np.nan
bakery.loc[5, "revenue"] = -45000  # 환불 또는 실수
bakery.loc[bakery.sample(10, random_state=1).index, "store_id"] = " S01 "  # 공백
bakery.loc[bakery.sample(8, random_state=2).index, "item"] = "케익"        # 표기 혼재('케이크' vs '케익')
bakery = pd.concat([bakery, bakery.iloc[[0, 1, 2, 3]]], ignore_index=True)   # 중복 4건

print("빵셀러 데이터 생성 완료:", bakery.shape)
bakery.head()

빵셀러 데이터 생성 완료: (2164, 6)


,store_id,date_str,item,quantity,unit_price,revenue
0,S01,20250401,쿠키,2,12000,24000.0
1,S02,2025-04-01,케이크,2,12000,24000.0
2,S03,2025-04-01,쿠키,1,5500,5500.0
3,S04,2025-06-20,쿠키,2,8500,17000.0
4,S05,20250401,샌드위치,5,4500,22500.0


## 1. 품질 점검 — `quality_report_full()`

컬럼별 타입·결측·고유값·공백 문제와 숫자 컬럼의 범위·음수·IQR 이상치를 한 표로 모읍니다. 데이터 전체 수준의 중복 건수는 별도로 출력합니다.

In [3]:
def quality_report_full(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    """컬럼별 품질 지표를 한 표로 반환하고, 데이터 전체 요약을 출력한다."""
    rows = []
    for col in df.columns:
        s = df[col]
        is_num = pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)
        info = {
            "column": col,
            "dtype": str(s.dtype),
            "n_missing": int(s.isna().sum()),
            "missing_pct": round(s.isna().mean() * 100, 2),
            "n_unique": int(s.nunique(dropna=True)),
            "sample_values": list(s.dropna().unique()[:5]),
        }
        if is_num:
            q1, q3 = s.quantile([0.25, 0.75])
            iqr = q3 - q1
            info.update({
                "min": s.min(),
                "max": s.max(),
                "n_negative": int((s < 0).sum()),
                "n_iqr_outlier": int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()),
                "n_whitespace": np.nan,
            })
        else:
            st = s.dropna().astype(str)
            info.update({
                "min": np.nan, "max": np.nan, "n_negative": np.nan, "n_iqr_outlier": np.nan,
                "n_whitespace": int((st != st.str.strip()).sum()),
            })
        rows.append(info)

    report = pd.DataFrame(rows).set_index("column")
    print(f"[{name}] 행 {len(df):,} / 열 {df.shape[1]}")
    print(f"[{name}] 완전 중복 행(duplicated): {df.duplicated().sum():,}건")
    print(f"[{name}] 결측이 하나라도 있는 행: {df.isna().any(axis=1).sum():,}건")
    return report

raw_report = quality_report_full(bakery, "bakery 원본")
raw_report

[bakery 원본] 행 2,164 / 열 6
[bakery 원본] 완전 중복 행(duplicated): 321건
[bakery 원본] 결측이 하나라도 있는 행: 60건


,dtype,n_missing,missing_pct,n_unique,sample_values,min,max,n_negative,n_iqr_outlier,n_whitespace
column,,,,,,,,,,
store_id,str,0,0.00,7,"[S01, S02, S03, S04, S05]",NaN,NaN,NaN,NaN,10.0
date_str,str,0,0.00,9,"[20250401, 2025-04-01, 2025-06-20, 2025/05/15,...",NaN,NaN,NaN,NaN,0.0
item,str,0,0.00,6,"[쿠키, 케이크, 샌드위치, 크루아상, 식빵]",NaN,NaN,NaN,NaN,0.0
quantity,int64,0,0.00,5,"[2, 1, 5, 3, 50]",1.0,50.0,0.0,315.0,NaN
unit_price,int64,0,0.00,5,"[12000, 5500, 8500, 4500, 3500]",3500.0,12000.0,0.0,0.0,NaN
revenue,float64,60,2.77,26,"[24000.0, 5500.0, 17000.0, 22500.0, -45000.0]",-45000.0,600000.0,1.0,364.0,NaN


In [4]:
# 범주형 컬럼의 표기 확인
for col in ["store_id", "item"]:
    print(col, "→", bakery[col].value_counts().to_dict())

# 날짜 문자열 형식 확인
fmt = bakery["date_str"].str.replace(r"\d", "9", regex=True)
print("date_str 형식 →", fmt.value_counts().to_dict())
print("실제 날짜 종류 →", sorted(bakery["date_str"].str.replace(r"[-/]", "", regex=True).unique()))

# 의심 수량
print("quantity 분포 →", bakery["quantity"].value_counts().sort_index().to_dict())

store_id → {'S02': 361, 'S04': 361, 'S01': 360, 'S03': 358, 'S05': 357, 'S06': 357, ' S01 ': 10}
item → {'쿠키': 446, '샌드위치': 443, '식빵': 435, '케이크': 417, '크루아상': 415, '케익': 8}
date_str 형식 → {'99999999': 747, '9999-99-99': 713, '9999/99/99': 704}
실제 날짜 종류 → ['20250401', '20250515', '20250620']
quantity 분포 → {1: 609, 2: 619, 3: 343, 5: 278, 50: 315}


**중복에 대한 판단.** `duplicated()`는 321건을 가리키지만, 이 데이터에는 시간대나 거래 ID 같은 식별 키가 없습니다. 매장·날짜(9종 문자열)·상품(5종)·수량·단가 조합이 적어 서로 다른 시간대 기록이 우연히 같아지는 경우가 많습니다. 생성 코드상 진짜 중복은 `concat`으로 끝에 다시 붙인 4건뿐이므로, 정제에서는 **원본 범위(`n_rows`) 밖에 있으면서 앞의 행과 완전히 같은 행**만 중복으로 제거합니다.

In [5]:
dup_all = bakery.duplicated(keep="first")
dup_appended = dup_all & (bakery.index >= n_rows)
print("duplicated() 전체:", dup_all.sum(), "건")
print("재삽입된 중복(원본 범위 밖):", dup_appended.sum(), "건")

duplicated() 전체: 321 건
재삽입된 중복(원본 범위 밖): 4 건


## 2. 정제 파이프라인 — 작은 함수 + `.pipe()`

각 단계 함수는 DataFrame을 받아 DataFrame을 돌려주고, 처리 전후 행 수와 영향 건수를 `decision_log`에 기록합니다. 음수 매출은 버리지 않고 `quarantine` 딕셔너리에 따로 보관합니다.

In [6]:
decision_log = []   # 단계별 결정 기록
quarantine = {}     # 집계에서 제외하되 보관할 행

def _log(step, problem, decision, reason, affected, before, after):
    decision_log.append({
        "단계": step, "문제": problem, "결정": decision, "근거": reason,
        "영향 행 수": affected, "처리 전 행": before, "처리 후 행": after,
    })


def drop_appended_duplicates(df, n_original):
    """원본 범위 밖에서 앞 행과 완전히 같은 행만 제거한다."""
    before = len(df)
    mask = df.duplicated(keep="first") & (df.index >= n_original)
    out = df.loc[~mask].reset_index(drop=True)
    _log("1. 중복 제거", f"완전 중복 {mask.sum()}건 재삽입 (duplicated() 전체는 {df.duplicated().sum()}건)",
         "재삽입된 행만 제거",
         "식별 키가 없어 내용이 같은 나머지 행은 다른 시간대의 정상 거래일 수 있음",
         int(mask.sum()), before, len(out))
    return out


def strip_store_id(df):
    """store_id 앞뒤 공백 제거."""
    n = int((df["store_id"] != df["store_id"].str.strip()).sum())
    out = df.assign(store_id=df["store_id"].str.strip())
    _log("2. store_id 공백", f"' S01 ' {n}건", "str.strip()으로 'S01' 통일",
         "공백 때문에 같은 매장이 다른 매장으로 집계되는 것을 방지", n, len(df), len(out))
    return out


def normalize_item(df, mapping):
    """상품명 표기 통일."""
    n = int(df["item"].isin(mapping.keys()).sum())
    out = df.assign(item=df["item"].replace(mapping))
    _log("3. item 표기 통일", f"'케익' {n}건", "'케익' → '케이크'",
         "같은 상품의 표기 차이로, 두면 상품별 매출이 쪼개짐", n, len(df), len(out))
    return out


def parse_dates(df):
    """세 가지 형식의 date_str을 datetime으로 변환하고 month 컬럼 추가."""
    cleaned = df["date_str"].str.replace(r"[-/]", "", regex=True)
    date = pd.to_datetime(cleaned, format="%Y%m%d", errors="coerce")
    n_fail = int(date.isna().sum())
    out = df.assign(date=date, month=date.dt.to_period("M").astype(str)).drop(columns="date_str")
    _log("4. 날짜 변환", "date_str 형식 3종 혼재 (YYYY-MM-DD, YYYY/MM/DD, YYYYMMDD)",
         "구분자 제거 후 %Y%m%d로 일괄 변환", f"형식만 다르고 값은 같은 날짜임 (변환 실패 {n_fail}건)",
         int(len(df)), len(df), len(out))
    return out


def split_negative_revenue(df, store):
    """음수 매출은 별도 보관하고 본 데이터에서 제외."""
    neg = df["revenue"] < 0
    store["negative_revenue"] = df.loc[neg].copy()
    out = df.loc[~neg].reset_index(drop=True)
    _log("5. 음수 매출", f"revenue 음수 {neg.sum()}건 (-45,000원)", "quarantine에 별도 보관 후 집계 제외",
         "환불인지 입력 오류인지 확인 전까지 매출 KPI를 왜곡하지 않도록", int(neg.sum()), len(df), len(out))
    return out


def drop_missing_revenue(df):
    """revenue 결측 행 제거."""
    na = df["revenue"].isna()
    out = df.loc[~na].reset_index(drop=True)
    _log("6. 매출 결측", f"revenue 결측 {na.sum()}건 (원본 대비 {na.sum() / len(bakery):.2%})", "행 제거",
         "비중이 작고, quantity×unit_price로 복원은 가능하나 결측 원인이 불명확해 임의 복원을 피함",
         int(na.sum()), len(df), len(out))
    return out


def flag_bulk_quantity(df, threshold=50):
    """의심 수량은 지우지 않고 플래그만 단다."""
    flag = df["quantity"] >= threshold
    out = df.assign(is_bulk_suspect=flag)
    _log("7. 의심 수량", f"quantity={threshold} {flag.sum()}건", "삭제하지 않고 is_bulk_suspect 플래그",
         "단체 주문일 수 있어 근거 없이 지우지 않음. 단, KPI 영향이 커서 별도 표기",
         int(flag.sum()), len(df), len(out))
    return out

In [7]:
decision_log.clear()
quarantine.clear()

bakery_clean = (
    bakery
    .pipe(drop_appended_duplicates, n_original=n_rows)
    .pipe(strip_store_id)
    .pipe(normalize_item, mapping={"케익": "케이크"})
    .pipe(parse_dates)
    .pipe(split_negative_revenue, store=quarantine)
    .pipe(drop_missing_revenue)
    .pipe(flag_bulk_quantity, threshold=50)
    .astype({"store_id": "category", "item": "category", "revenue": "int64"})
)

print("원본:", bakery.shape, "→ 정제 후:", bakery_clean.shape)
bakery_clean.head()

원본: (2164, 6) → 정제 후: (2099, 8)


,store_id,item,quantity,unit_price,revenue,date,month,is_bulk_suspect
0,S01,쿠키,2,12000,24000,2025-04-01,2025-04,False
1,S02,케이크,2,12000,24000,2025-04-01,2025-04,False
2,S03,쿠키,1,5500,5500,2025-04-01,2025-04,False
3,S04,쿠키,2,8500,17000,2025-06-20,2025-06,False
4,S05,샌드위치,5,4500,22500,2025-04-01,2025-04,False


In [8]:
# 정제 결과 재점검
clean_report = quality_report_full(bakery_clean, "bakery_clean")
print("store_id →", sorted(bakery_clean["store_id"].unique()))
print("item →", sorted(bakery_clean["item"].unique()))
print("보관된 음수 매출:")
display(quarantine["negative_revenue"])
clean_report

[bakery_clean] 행 2,099 / 열 8
[bakery_clean] 완전 중복 행(duplicated): 763건
[bakery_clean] 결측이 하나라도 있는 행: 0건
store_id → ['S01', 'S02', 'S03', 'S04', 'S05', 'S06']
item → ['샌드위치', '식빵', '케이크', '쿠키', '크루아상']
보관된 음수 매출:


,store_id,item,quantity,unit_price,revenue,date,month
5,S06,샌드위치,5,4500,-45000.0,2025-05-15,2025-05


,dtype,n_missing,missing_pct,n_unique,sample_values,min,max,n_negative,n_iqr_outlier,n_whitespace
column,,,,,,,,,,
store_id,category,0,0.0,6,"[S01, S02, S03, S04, S05]",NaN,NaN,NaN,NaN,0.0
item,category,0,0.0,5,"[쿠키, 케이크, 샌드위치, 크루아상, 식빵]",NaN,NaN,NaN,NaN,0.0
quantity,int64,0,0.0,5,"[2, 1, 5, 3, 50]",1.0,50.0,0.0,308.0,NaN
unit_price,int64,0,0.0,5,"[12000, 5500, 8500, 4500, 3500]",3500.0,12000.0,0.0,0.0,NaN
revenue,int64,0,0.0,25,"[24000, 5500, 17000, 22500, 9000]",3500.0,600000.0,0.0,363.0,NaN
date,datetime64[us],0,0.0,3,"[2025-04-01 00:00:00, 2025-06-20 00:00:00, 202...",NaN,NaN,NaN,NaN,0.0
month,str,0,0.0,3,"[2025-04, 2025-06, 2025-05]",NaN,NaN,NaN,NaN,0.0
is_bulk_suspect,bool,0,0.0,2,"[False, True]",NaN,NaN,NaN,NaN,0.0


## 3. 결정 로그 표

In [9]:
decision_df = pd.DataFrame(decision_log)

def df_to_markdown(df):
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |",
             "|" + "|".join(["---"] * len(cols)) + "|"]
    for row in df.astype(str).itertuples(index=False):
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines)

print(df_to_markdown(decision_df))

| 단계 | 문제 | 결정 | 근거 | 영향 행 수 | 처리 전 행 | 처리 후 행 |
|---|---|---|---|---|---|---|
| 1. 중복 제거 | 완전 중복 4건 재삽입 (duplicated() 전체는 321건) | 재삽입된 행만 제거 | 식별 키가 없어 내용이 같은 나머지 행은 다른 시간대의 정상 거래일 수 있음 | 4 | 2164 | 2160 |
| 2. store_id 공백 | ' S01 ' 10건 | str.strip()으로 'S01' 통일 | 공백 때문에 같은 매장이 다른 매장으로 집계되는 것을 방지 | 10 | 2160 | 2160 |
| 3. item 표기 통일 | '케익' 8건 | '케익' → '케이크' | 같은 상품의 표기 차이로, 두면 상품별 매출이 쪼개짐 | 8 | 2160 | 2160 |
| 4. 날짜 변환 | date_str 형식 3종 혼재 (YYYY-MM-DD, YYYY/MM/DD, YYYYMMDD) | 구분자 제거 후 %Y%m%d로 일괄 변환 | 형식만 다르고 값은 같은 날짜임 (변환 실패 0건) | 2160 | 2160 | 2160 |
| 5. 음수 매출 | revenue 음수 1건 (-45,000원) | quarantine에 별도 보관 후 집계 제외 | 환불인지 입력 오류인지 확인 전까지 매출 KPI를 왜곡하지 않도록 | 1 | 2160 | 2159 |
| 6. 매출 결측 | revenue 결측 60건 (원본 대비 2.77%) | 행 제거 | 비중이 작고, quantity×unit_price로 복원은 가능하나 결측 원인이 불명확해 임의 복원을 피함 | 60 | 2159 | 2099 |
| 7. 의심 수량 | quantity=50 308건 | 삭제하지 않고 is_bulk_suspect 플래그 | 단체 주문일 수 있어 근거 없이 지우지 않음. 단, KPI 영향이 커서 별도 표기 | 308 | 2099 | 2099 |


**결정 로그 (위 출력을 마크다운으로 고정)**

| 단계              | 문제                                                      | 결정                                 | 근거                                                                                     |   영향 행 수 |   처리 전 행 |   처리 후 행 |
|:------------------|:----------------------------------------------------------|:-------------------------------------|:-----------------------------------------------------------------------------------------|-------------:|-------------:|-------------:|
| 1. 중복 제거      | 완전 중복 4건 재삽입 (duplicated() 전체는 321건)          | 재삽입된 행만 제거                   | 식별 키가 없어 내용이 같은 나머지 행은 다른 시간대의 정상 거래일 수 있음                 |            4 |         2164 |         2160 |
| 2. store_id 공백  | ' S01 ' 10건                                              | str.strip()으로 'S01' 통일           | 공백 때문에 같은 매장이 다른 매장으로 집계되는 것을 방지                                 |           10 |         2160 |         2160 |
| 3. item 표기 통일 | '케익' 8건                                                | '케익' → '케이크'                    | 같은 상품의 표기 차이로, 두면 상품별 매출이 쪼개짐                                       |            8 |         2160 |         2160 |
| 4. 날짜 변환      | date_str 형식 3종 혼재 (YYYY-MM-DD, YYYY/MM/DD, YYYYMMDD) | 구분자 제거 후 %Y%m%d로 일괄 변환    | 형식만 다르고 값은 같은 날짜임 (변환 실패 0건)                                           |         2160 |         2160 |         2160 |
| 5. 음수 매출      | revenue 음수 1건 (-45,000원)                              | quarantine에 별도 보관 후 집계 제외  | 환불인지 입력 오류인지 확인 전까지 매출 KPI를 왜곡하지 않도록                            |            1 |         2160 |         2159 |
| 6. 매출 결측      | revenue 결측 60건 (원본 대비 2.77%)                       | 행 제거                              | 비중이 작고, quantity×unit_price로 복원은 가능하나 결측 원인이 불명확해 임의 복원을 피함 |           60 |         2159 |         2099 |
| 7. 의심 수량      | quantity=50 308건                                         | 삭제하지 않고 is_bulk_suspect 플래그 | 단체 주문일 수 있어 근거 없이 지우지 않음. 단, KPI 영향이 커서 별도 표기                 |          308 |         2099 |         2099 |

## 4. KPI 집계

- **월·매장 KPI**: 매출 합계, 거래 건수, 판매 수량, 건당 평균 매출, 의심 수량 비중
- **상품 KPI**: 매출 합계, 거래 건수, 판매 수량, 매출 비중, 의심 수량 제외 매출

In [10]:
kpi_month_store = (
    bakery_clean
    .groupby(["month", "store_id"], observed=True)
    .agg(revenue=("revenue", "sum"),
         orders=("revenue", "size"),
         quantity=("quantity", "sum"),
         bulk_orders=("is_bulk_suspect", "sum"))
    .assign(avg_order_value=lambda d: (d["revenue"] / d["orders"]).round(0),
            bulk_order_pct=lambda d: (d["bulk_orders"] / d["orders"] * 100).round(1))
    .reset_index()
    .sort_values("revenue", ascending=False, ignore_index=True)
)
kpi_month_store.head(10)

,month,store_id,revenue,orders,quantity,bulk_orders,avg_order_value,bulk_order_pct
0,2025-04,S06,10391000,121,1492,25,85876.0,20.7
1,2025-04,S05,9599000,125,1510,26,76792.0,20.8
2,2025-06,S02,8848000,118,1227,20,74983.0,16.9
3,2025-04,S01,8666000,122,1253,20,71033.0,16.4
4,2025-06,S03,8612000,102,1245,21,84431.0,20.6
5,2025-05,S04,8381000,127,1174,18,65992.0,14.2
6,2025-04,S03,8303500,115,1248,21,72204.0,18.3
7,2025-05,S01,8137500,108,1165,19,75347.0,17.6
8,2025-04,S02,7771500,115,1067,17,67578.0,14.8
9,2025-04,S04,7486500,103,955,15,72684.0,14.6


In [11]:
kpi_item = (
    bakery_clean
    .groupby("item", observed=True)
    .agg(revenue=("revenue", "sum"),
         orders=("revenue", "size"),
         quantity=("quantity", "sum"))
    .assign(revenue_share_pct=lambda d: (d["revenue"] / d["revenue"].sum() * 100).round(1))
    .join(bakery_clean.loc[~bakery_clean["is_bulk_suspect"]]
          .groupby("item", observed=True)["revenue"].sum().rename("revenue_excl_bulk"))
    .reset_index()
    .sort_values("revenue", ascending=False, ignore_index=True)
)
kpi_item

,item,revenue,orders,quantity,revenue_share_pct,revenue_excl_bulk
0,쿠키,30436000,435,4059,22.6,5911000
1,샌드위치,29373000,435,4275,21.8,5648000
2,식빵,27046000,422,4257,20.0,5421000
3,케이크,25331500,410,3593,18.8,5406500
4,크루아상,22775500,397,3357,16.9,5550500


In [12]:
top_ms = kpi_month_store.iloc[0]
top_item = kpi_item.iloc[0]
bulk_rev_share = bakery_clean.loc[bakery_clean["is_bulk_suspect"], "revenue"].sum() / bakery_clean["revenue"].sum()

print(f"월·매장 1위: {top_ms['month']} {top_ms['store_id']} — {top_ms['revenue']:,}원")
print(f"상품 1위: {top_item['item']} — {top_item['revenue']:,}원 (비중 {top_item['revenue_share_pct']}%)")
print(f"의심 수량(50) 행 비중 {bakery_clean['is_bulk_suspect'].mean():.1%}, 매출 비중 {bulk_rev_share:.1%}")

월·매장 1위: 2025-04 S06 — 10,391,000원
상품 1위: 쿠키 — 30,436,000원 (비중 22.6%)
의심 수량(50) 행 비중 14.7%, 매출 비중 79.3%


## 5. 저장 — Parquet과 CSV 용량 비교

In [13]:
# KPI Parquet 2개
kpi_month_store.to_parquet(OUT / "kpi_month_store.parquet", index=False)
kpi_item.to_parquet(OUT / "kpi_item.parquet", index=False)

# 정제 데이터: Parquet과 CSV
bakery_clean.to_parquet(OUT / "bakery_clean.parquet", index=False)
bakery_clean.to_csv(OUT / "bakery_clean.csv", index=False, encoding="utf-8-sig")

pq_size = (OUT / "bakery_clean.parquet").stat().st_size
csv_size = (OUT / "bakery_clean.csv").stat().st_size
print(f"bakery_clean: Parquet {pq_size/1024:.1f}KB vs CSV {csv_size/1024:.1f}KB → Parquet이 CSV의 {pq_size/csv_size:.0%} 크기 ({1 - pq_size/csv_size:.0%} 절감)")

bakery_clean: Parquet 10.7KB vs CSV 106.6KB → Parquet이 CSV의 10% 크기 (90% 절감)


In [14]:
# 다시 읽어서 타입이 보존되는지 확인
back_pq = pd.read_parquet(OUT / "bakery_clean.parquet")
back_csv = pd.read_csv(OUT / "bakery_clean.csv")
pd.DataFrame({"parquet": back_pq.dtypes.astype(str), "csv": back_csv.dtypes.astype(str)})

,parquet,csv
store_id,category,str
item,category,str
quantity,int64,int64
unit_price,int64,int64
revenue,int64,int64
date,datetime64[us],str
month,str,str
is_bulk_suspect,bool,bool


CSV로 다시 읽으면 `date`는 문자열, `store_id`·`item`은 일반 문자열로 돌아오지만, Parquet은 datetime과 category 타입을 그대로 보존합니다.

## 6. 종합 프로젝트 보고서

### 1. 데이터 개요
- 출처: (가상) 빵셀러 베이커리 체인 운영 로그 (6개 매장, 5개 상품)
- 기간: 실제 날짜는 2025-04-01, 2025-05-15, 2025-06-20 세 날뿐 (3개월 연속 데이터가 아님)
- 원본 행 수 / 정제 후 행 수: 2,164행 → 2,099행 (65행 제외: 중복 4, 음수 1, 결측 60)

### 2. 발견한 품질 문제
- 결측: revenue 60건 (2.77%), 나머지 컬럼은 결측 없음
- 중복: 재삽입된 완전 중복 4건. `duplicated()`는 321건을 가리키지만 식별 키가 없어 대부분 정상 거래로 판단
- 이상치: revenue 음수 1건(-45,000원), quantity=50 315건(원본의 14.6%)
- 표기 혼재: store_id `' S01 '` 공백 10건, item `'케익'` 8건, date_str 형식 3종(YYYY-MM-DD / YYYY/MM/DD / YYYYMMDD)

### 3. 처리 결정과 근거 (5줄로)
1. 중복 행 4건 → 원본 범위 밖에 재삽입된 행만 제거 (내용이 같은 나머지 행은 다른 시간대의 정상 거래일 수 있음)
2. store_id 공백 → `str.strip()`으로 `'S01'` 통일 (같은 매장이 따로 집계되는 것을 방지)
3. item '케익' → '케이크' → 같은 상품의 표기 차이이므로 통일 (상품별 매출 분산 방지)
4. revenue 음수 → 별도 보관 1건 (`quarantine["negative_revenue"]`, 환불·오류 확인 전까지 KPI에서 제외)
5. revenue 결측 60건 → 제거 (이유: 비중 2.77%로 작고, quantity×unit_price로 복원할 수는 있지만 결측 원인이 불명확해 임의 복원을 피함)

### 4. 주요 KPI 결과 (2줄)
- (월·매장 관점) 가장 매출이 큰 매장과 그 월: 2025년 4월 S06, 10,391,000원 (2위 4월 S05 9,599,000원)
- (상품 관점) 매출 1위 상품: 쿠키, 30,436,000원 (전체의 22.6%, 2위 샌드위치 29,373,000원)

### 5. 한계와 후속 작업
- quantity=50 행이 정제 데이터의 14.7%인데 매출의 79.3%를 차지함. 이 행을 빼면 상품 2위가 샌드위치에서 크루아상으로 바뀌므로, 단체 주문인지 입력 오류인지 확인이 최우선
- 시간대·거래 ID 컬럼이 없어 진짜 중복과 정상적인 동일 거래를 구분할 수 없음. 원천 시스템에서 키 컬럼 확보 필요
- 날짜가 3일뿐이라 '월별' 비교는 사실상 하루 단위 비교임. 기간 전체 로그 확보 필요
- 환불 데이터(음수 매출)의 발생 원인 분석과 별도 집계 기준 마련 필요
- '케익' 외 다른 표기 변종(예: 띄어쓰기, 영문 표기)이 신규 데이터에 들어오는지 정기 점검 필요
- 저장 형식: 정제 데이터는 Parquet 10.7KB, CSV 106.6KB로 Parquet이 약 90% 작고 타입(datetime·category)도 보존됨